# Phase 6 — Fraud Decision Engine

This notebook separates **model score**, **binary classification**, **assumed business cost**, and **risk band**. It uses the Phase 5 score exports so that threshold selection is performed on the validation partition only and evaluated afterward on the test partition.

**Important:** XGBoost was trained with class weighting and has not been calibrated in this phase. Treat `fraud_score` as an uncalibrated model score, not a verified real-world fraud probability.

## 1. The idea: score vs. classification

- A continuous score answers: *How strongly does the model rank this row as suspicious?*
- A threshold converts that score into a binary class: `score >= threshold` means flag for review.
- A risk score on a 0–100 scale is simply `100 × model_score`; it is **not** proof of a calibrated probability percentage.
- Threshold selection changes precision and recall without retraining the model.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
assert (ROOT / "src").exists(), "Start Jupyter from the fraudshield-ai project root or its notebooks folder."
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import numpy as np
from IPython.display import display
from src.evaluation.decision_engine import (
    load_policy_config, load_prediction_file, threshold_sweep, choose_cost_threshold,
    threshold_for_target_recall, evaluate_threshold, assign_risk_level, run_decision_engine,
)

CONFIG_PATH = ROOT / "configs" / "decision_policy.yaml"
VAL_PATH = ROOT / "reports" / "xgboost" / "validation_predictions.csv"
TEST_PATH = ROOT / "reports" / "xgboost" / "test_predictions.csv"
config = load_policy_config(CONFIG_PATH)
val_df = load_prediction_file(VAL_PATH, "validation")
test_df = load_prediction_file(TEST_PATH, "test")
print("Validation rows/fraud:", len(val_df), int(val_df.actual_class.sum()))
print("Test rows/fraud:", len(test_df), int(test_df.actual_class.sum()))
print("Assumed relative costs:", config["costs"])

Validation rows/fraud: 56958 57
Test rows/fraud: 56961 75
Assumed relative costs: {'false_positive': 1, 'false_negative': 100}


## 2. Select a threshold on validation only

We define a simple, transparent cost function:

`total_cost = false_positive_cost × FP + false_negative_cost × FN`

The default policy assumes FP cost = 1 and FN cost = 100 **relative units**. Those values are illustrative and are not measured financial losses. The selected threshold minimizes this assumed cost on validation data. The test labels are not used to pick the threshold.

In [2]:
y_val = val_df["actual_class"].to_numpy(dtype=int)
s_val = val_df["fraud_score"].to_numpy(dtype=float)
y_test = test_df["actual_class"].to_numpy(dtype=int)
s_test = test_df["fraud_score"].to_numpy(dtype=float)
c_fp = float(config["costs"]["false_positive"])
c_fn = float(config["costs"]["false_negative"])

sweep = threshold_sweep(y_val, s_val, cost_fp=c_fp, cost_fn=c_fn)
chosen = choose_cost_threshold(sweep)
selected_threshold = float(chosen["threshold"])
print(f"Validation-selected threshold: {selected_threshold:.8f}")
display(chosen.to_frame("selected_validation_row"))

Validation-selected threshold: 0.78530025


,selected_validation_row
threshold,0.785300
true_negative,56871.000000
false_positive,30.000000
false_negative,11.000000
true_positive,46.000000
predicted_positive,76.000000
precision,0.605263
recall,0.807018
f1,0.691729
cost_fp_weighted,30.000000


## 3. Compare default and cost-selected thresholds

Average Precision and ROC-AUC evaluate ranking across thresholds; precision, recall, F1, and confusion counts depend on the selected threshold. We report both types.

In [3]:
rows = []
for name, threshold in [("Default 0.5", 0.5), ("Validation-selected cost threshold", selected_threshold)]:
    for part_name, y, scores in [("Validation", y_val, s_val), ("Test", y_test, s_test)]:
        row = evaluate_threshold(y, scores, threshold, c_fp, c_fn)
        row["partition"] = part_name
        row["policy"] = name
        rows.append(row)
comparison = pd.DataFrame(rows)[["partition", "policy", "threshold", "precision", "recall", "f1",
                                 "average_precision", "roc_auc", "false_positive", "false_negative", "total_cost_units"]]
display(comparison.round(5))

,partition,policy,threshold,precision,recall,f1,average_precision,roc_auc,false_positive,false_negative,total_cost_units
0,Validation,Default 0.5,0.5000,0.16312,0.80702,0.27139,0.77951,0.97399,236,11,1336.0
1,Test,Default 0.5,0.5000,0.26316,0.80000,0.39604,0.78718,0.98568,168,15,1668.0
2,Validation,Validation-selected cost threshold,0.7853,0.60526,0.80702,0.69173,0.77951,0.97399,30,11,1130.0
3,Test,Validation-selected cost threshold,0.7853,0.76316,0.77333,0.76821,0.78718,0.98568,18,17,1718.0


## 4. Cost sensitivity

The best threshold depends on relative costs. Change the configured false-negative cost and re-run the sweep to see how alert volume, false positives, and missed fraud change. A cost ratio should reflect documented business assumptions, not be selected because it produces the prettiest metric.

In [4]:
sensitivity_rows = []
for fn_cost in config.get("cost_sensitivity", {}).get("false_negative_costs", [1, 5, 10, 25, 50, 100, 250, 500]):
    candidate_sweep = threshold_sweep(y_val, s_val, cost_fp=c_fp, cost_fn=float(fn_cost))
    best = choose_cost_threshold(candidate_sweep)
    th = float(best["threshold"])
    vm = evaluate_threshold(y_val, s_val, th, c_fp, float(fn_cost))
    tm = evaluate_threshold(y_test, s_test, th, c_fp, float(fn_cost))
    sensitivity_rows.append({"FN_cost_units": fn_cost, "FP_cost_units": c_fp, "threshold": th,
                             "val_FP": vm["false_positive"], "val_FN": vm["false_negative"],
                             "val_cost": vm["total_cost_units"], "test_FP": tm["false_positive"],
                             "test_FN": tm["false_negative"], "test_cost": tm["total_cost_units"]})
display(pd.DataFrame(sensitivity_rows))

,FN_cost_units,FP_cost_units,threshold,val_FP,val_FN,val_cost,test_FP,test_FN,test_cost
0,1,1.0,0.968864,0,21,21.0,2,32,34.0
1,5,1.0,0.835528,15,13,80.0,10,19,105.0
2,10,1.0,0.785300,30,11,140.0,18,17,188.0
3,25,1.0,0.785300,30,11,305.0,18,17,443.0
4,50,1.0,0.785300,30,11,580.0,18,17,868.0
5,100,1.0,0.785300,30,11,1130.0,18,17,1718.0
6,250,1.0,0.293915,553,8,2553.0,319,11,3069.0
7,500,1.0,0.293915,553,8,4553.0,319,11,5819.0


## 5. Assign relative risk bands

For the starter policy, `LOW` and `MEDIUM` boundaries are anchored to validation score cutoffs that achieve at least 95% and 85% recall, respectively. `CRITICAL` begins at the cost-selected decision threshold; `HIGH` is the band between the 85%-recall cutoff and the selected threshold. These are **relative triage bands**, not probability intervals.

The bands and costs are configurable. Before real use, review them against alert capacity and independently validated cost estimates.

In [5]:
low_anchor = threshold_for_target_recall(sweep, float(config["risk_bands"]["target_recall_low_medium"]))
medium_anchor = threshold_for_target_recall(sweep, float(config["risk_bands"]["target_recall_medium_high"]))
low_t = float(low_anchor["threshold"])
medium_t = float(medium_anchor["threshold"])
assert low_t < medium_t < selected_threshold

val_bands = assign_risk_level(s_val, low_t, medium_t, selected_threshold)
test_bands = assign_risk_level(s_test, low_t, medium_t, selected_threshold)
print({"LOW/MEDIUM boundary": low_t, "MEDIUM/HIGH boundary": medium_t, "HIGH/CRITICAL boundary": selected_threshold})
display(pd.DataFrame({"validation_count": pd.Series(val_bands).value_counts(),
                      "test_count": pd.Series(test_bands).value_counts()}).reindex(["LOW", "MEDIUM", "HIGH", "CRITICAL"]).fillna(0).astype(int))

{'LOW/MEDIUM boundary': 0.027192619, 'MEDIUM/HIGH boundary': 0.29391524, 'HIGH/CRITICAL boundary': 0.78530025}


,validation_count,test_count
LOW,41080,44966
MEDIUM,15276,11612
HIGH,526,307
CRITICAL,76,76


## 6. Generate the complete reproducible artifact set

This runs the Phase 6 module and writes the threshold sweep, cost sensitivity, per-row decisions, policy JSON, confusion matrix, and plots. It never uses the test set to select the threshold.

In [6]:
result = run_decision_engine(VAL_PATH, TEST_PATH, CONFIG_PATH,
                             ROOT / "reports" / "decision_engine",
                             ROOT / "models" / "decision_policy.json")
print(json.dumps({
    "selected_threshold": result["selected_threshold"],
    "validation_cost_default": result["validation_default"]["total_cost_units"],
    "validation_cost_selected": result["validation_selected"]["total_cost_units"],
    "test_cost_default": result["test_default"]["total_cost_units"],
    "test_cost_selected": result["test_selected"]["total_cost_units"],
}, indent=2))

{
  "selected_threshold": 0.78530025,
  "validation_cost_default": 1336.0,
  "validation_cost_selected": 1130.0,
  "test_cost_default": 1668.0,
  "test_cost_selected": 1718.0
}


## 7. Interpret honestly

The current default cost scenario lowers cost on validation but **does not lower the assumed cost on the held-out test period**. That is not a reason to tune directly against test labels. It is evidence that threshold selection is uncertain and the score/distribution changes over time.

For a production-style policy, the next steps are to: (1) define costs and alert capacity with domain stakeholders, (2) use additional temporal validation windows, (3) assess calibration separately before calling the score a literal probability, and (4) monitor delayed ground-truth labels after deployment.